# Diagnose a real Qwen model on SageMaker with vLLM

Upload this notebook to a SageMaker Studio GPU JupyterLab space and run the cells in order. It includes the experiment code. It downloads the public **Qwen2.5-0.5B-Instruct** weights and starts a private native vLLM server on each selected GPU. Use a single host with **two or four Ampere-or-newer GPUs**. An `ml.g5.12xlarge` has four A10G GPUs.

The defaults run a handoff pilot followed by one repetition of the study. Service collection takes about 72 minutes on two GPUs or 80 minutes on four, plus setup, calibration and evaluation. The notebook uses the GPUs assigned to this space. The space continues to accrue its usual charges while running. Outputs remain under `dat-vllm-experiment` in the notebook directory.

## 1. Unpack the included source

The cell contains a compressed copy of the Python package. The printed directory holds readable source files.

In [1]:
from pathlib import Path
import base64, csv, hashlib, io, json, os, shutil, subprocess, sys, venv, zipfile
from IPython.display import display
import pandas as pd

root = (Path.cwd() / 'dat-vllm-experiment').resolve()
root.mkdir(exist_ok=True)
code_dir = root / 'source-50bbfc2c3e'
code_dir.mkdir(exist_ok=True)
payload = base64.b64decode('...')
if hashlib.sha256(payload).hexdigest() != '50bbfc2c3ecf8107832c00e5d32d829a2d81ffe6abeca6b44cc688f64bc74eb4':
    raise ValueError('Embedded source integrity check failed')
with zipfile.ZipFile(io.BytesIO(payload)) as source:
    source.extractall(code_dir)
print('Project directory', root)

Project directory /home/sagemaker-user/DAT-CS5903/DAT_CS5903_Final/DAT-CS5903/dat-vllm-experiment


## 2. Choose the experiment and prepare the GPU runtime

Use `PILOT_ONLY = True` for just the token-equivalence and cache-handoff checks. For the full research configuration, use three repetitions and 180 seconds per trial. The setup uses an isolated Python environment and leaves the notebook kernel available to inspect results.

In [2]:
PILOT_ONLY = True
REPETITIONS = 1
SECONDS_PER_TRIAL = 120
MEASURE_OVERHEAD = False
REQUESTS_PER_SECOND = 2.6

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Select a SageMaker GPU JupyterLab space before running the experiment.')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,compute_cap,memory.total',
                '--format=csv'], check=True)
env_dir = root / 'venv-vllm-0102'
py = env_dir / 'bin/python'
probe = 'import sys; print(int((3,10) <= sys.version_info[:2] <= (3,12)))'
def compatible(exe):
    if not exe or not Path(exe).is_file():
        return False
    check = subprocess.run([str(exe), '-c', probe], capture_output=True, text=True)
    return check.returncode == 0 and check.stdout.strip() == '1'

if not compatible(py):
    candidates = [shutil.which(name) for name in ('python3.12', 'python3.11', 'python3.10')]
    interpreter = next((exe for exe in [sys.executable, *candidates] if compatible(exe)), None)
    if env_dir.exists():
        env_dir = root / 'venv-vllm-0102-compatible'
        py = env_dir / 'bin/python'
    if not compatible(py):
        if interpreter:
            subprocess.run([interpreter, '-m', 'venv', str(env_dir)], check=True)
        else:
            uv = shutil.which('uv')
            if not uv:
                tool_dir = root / 'bootstrap-tools'
                subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(tool_dir), 'uv'], check=True)
                uv = str(tool_dir / 'bin/uv')
            subprocess.run([uv, 'python', 'install', '3.12'], check=True)
            subprocess.run([uv, 'venv', '--python', '3.12', '--seed', str(env_dir)], check=True)
print('Experiment interpreter:', subprocess.check_output([str(py), '--version'], text=True).strip())
subprocess.run([str(py), '-m', 'pip', 'install', '-r', str(code_dir / 'requirements-vllm-gpu.txt')], check=True)
env = dict(os.environ, PYTHONPATH=str(code_dir), OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1')
hardware = json.loads(subprocess.check_output([str(py), '-c',
    'import torch,json;print(json.dumps([[torch.cuda.get_device_name(i),list(torch.cuda.get_device_capability(i))] for i in range(torch.cuda.device_count())]))'],
    text=True, env=env))
workers = 2 if len(hardware) >= 4 else 1
if len(hardware) < 2 or any(cap[0] < 8 for _, cap in hardware[:2*workers]):
    raise RuntimeError(f'Need two or four visible SM80+ GPUs. Found {hardware}')
print('Workers per stage', workers, '| Selected devices', hardware[:2*workers])

index, name, compute_cap, memory.total [MiB]
0, NVIDIA A10G, 8.6, 23028 MiB
1, NVIDIA A10G, 8.6, 23028 MiB
2, NVIDIA A10G, 8.6, 23028 MiB
3, NVIDIA A10G, 8.6, 23028 MiB


Installed Python 3.12.16 in 742ms
 + cpython-3.12.16-linux-x86_64-gnu (python3.12)
Using CPython 3.12.16
Creating virtual environment with seed packages at: dat-vllm-experiment/venv-vllm-0102
 + pip==26.3
Activate with: source dat-vllm-experiment/venv-vllm-0102/bin/activate


Experiment interpreter: Python 3.12.16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB 80.2 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 145.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 136.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 143.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 139.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 240.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 171.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 39.7 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 61.4 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 166.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 133.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 115.7 

## 3. Verify live serving and KV reuse

Each prefill/decode pair must produce exactly the same eight greedy token IDs as an unsplit request. The decode server records how many external KV tokens it loaded. Each HTTP token also needs a corresponding worker-side forward record. A failed gate stops the run. Server diagnostics are written to `server-logs/`.

In [3]:
pilot_dir = root / f'pilot-{workers}x{workers}'
def launch(command, output, extra=()):
    cmd = [str(py), '-m', 'failslow', command, '--backend', 'vllm-http',
           '--workers', str(workers), '--out', str(output), '--spool', str(root / 'kv-spool'), *extra]
    subprocess.run(cmd, cwd=code_dir, env=env, check=True)

launch('pilot', pilot_dir)
pilot = json.loads((pilot_dir / 'pilot.json').read_text())
df = pd.DataFrame(pilot['paths'])
display(df[['prefill','decode','loaded_tokens','bytes','exact_match','generated_text']])
print('Model', pilot['profile']['workers']['p1']['model'])
print('Worker version', pilot['profile']['workers']['p1']['vllm'])

{
  "config_id": "9de6741bc819124553d0",
  "profile": {
    "host": "default",
    "workers": {
      "p1": {
        "engine": "vllm-http",
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "revision": "7ae557604adf67be50417f59c2c2f167def9a775",
        "torch": "2.8.0+cu128",
        "cuda": "12.8",
        "device": "NVIDIA A10G",
        "device_uuid": "37f0a0a3-629d-2b44-7033-ac051490fee8",
        "visible_device": "0",
        "vocab": 151665,
        "batch_size": 1,
        "dtype": "float16",
        "block_size": 16,
        "workload": "chat-template-v1",
        "attention_backend": "FLASH_ATTN",
        "vllm": "0.10.2"
      },
      "p2": {
        "engine": "vllm-http",
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "revision": "7ae557604adf67be50417f59c2c2f167def9a775",
        "torch": "2.8.0+cu128",
        "cuda": "12.8",
        "device": "NVIDIA A10G",
        "device_uuid": "741f5d9e-bcf9-e75f-7b5d-0ab29df20080",
        "visible_device": "1",
     

,prefill,decode,loaded_tokens,bytes,exact_match,generated_text
0,p1,d1,128,1574976,True,Paris is the capital of France.
1,p1,d2,128,1574976,True,Paris is the capital of France.
2,p2,d1,128,1574976,True,Paris is the capital of France.
3,p2,d2,128,1574976,True,Paris is the capital of France.


Model Qwen/Qwen2.5-0.5B-Instruct
Worker version 0.10.2


## 4. Run the detector on live requests

The serving workers process tokenized natural-language prompts over `/v1/completions`. The experiment fits a baseline from healthy server observations and then evaluates matched controls, component slowdowns and visibility failures. Reusing this directory resumes complete trials with the same configuration.

In [4]:
study_dir = root / f'study-{workers}x{workers}-r{REPETITIONS}-t{SECONDS_PER_TRIAL}-q{REQUESTS_PER_SECOND:g}'
if not PILOT_ONLY:
    args = ['--reps', str(REPETITIONS), '--duration', str(SECONDS_PER_TRIAL),
            '--rate', str(REQUESTS_PER_SECOND)]
    if MEASURE_OVERHEAD:
        args.append('--with-overhead')
    launch('suite', study_dir, args)
    comparison = pd.read_csv(study_dir / 'summary.csv')
    display(comparison[['method','incidents','top1','top3','misses','false_episodes','healthy_hours']])
else:
    print('Pilot completed. The study runs when PILOT_ONLY is False.')

Pilot completed. The study runs when PILOT_ONLY is False.


## 5. Inspect actual generations, slowdowns and false alarms

First-choice accuracy counts every injected incident. An empty ranking is an abstention. Paired output checks use requests that completed before the observation horizon in both runs. Path timings include the file-backed KV copy on this host.

In [5]:
if not PILOT_ONLY:
    import gzip
    first_run = next(t['rid'] for t in json.loads((study_dir / 'study.json').read_text())['trials'] if t['target'] == 'd1')
    folder = study_dir / 'runs' / first_run
    with gzip.open(folder / 'generations.jsonl.gz', 'rt') as f:
        examples = [json.loads(line) for line in f]
    display(pd.DataFrame(examples).head(3))
    inc = pd.read_csv(study_dir / 'incidents.csv')
    display(inc[inc.method == 'conditioned'][['case','strength','target','first_ranking','first_correct_top1_delay_s']])
    display(pd.read_csv(study_dir / 'generation_pairs.csv'))
    if MEASURE_OVERHEAD:
        display(pd.read_csv(study_dir / 'overhead' / 'overhead_paired.csv'))
    print('Results directory', study_dir)